# 无人小车感知与控制平台

## 项目简介

本项目实现一个把超声波避障、红外循迹、编码器测速和视觉辅助控制整合在同一底盘上的无人小车感知与控制平台。系统通过模式切换，在同一套硬件上依次演示基础避障、线路跟随、速度闭环和视觉偏差输入控制，适合作为后续自动驾驶小车课设的底层实验平台。


## 主要器件

| 器件 | 数量 | 说明 |
| --- | --- | --- |
| Arduino Uno | 1 | 主控板 |
| HC-SR04 | 1 | 前向避障测距 |
| 三路红外循迹模块 | 1 组 | 路径跟随 |
| 编码器 | 1 路 | 轮速估计 |
| 摄像头 / 上位机视觉模块 | 1 套 | 视觉偏差输入 |
| 双路电机驱动 + 底盘 | 1 套 | 差速驱动 |
| OLED | 1 | 状态显示 |


## 核心功能

- 通过按键切换巡线、避障、速度保持和视觉辅助四种模式。
- 超声波负责前向障碍检测，红外负责黑线状态检测。
- 编码器用于估算转速，演示基础速度闭环。
- 视觉模块通过串口下发横向偏差，小车端完成差速修正。
- OLED 实时显示模式、距离、红外状态、转速和视觉误差。


## 引脚分配

| 模块 | 引脚 |
| --- | --- |
| HC-SR04 Trig / Echo | D12、D13 |
| 红外循迹左 / 中 / 右 | A0、A1、A2 |
| 编码器输入 | D2 |
| 模式切换按键 | D4 |
| 左电机 PWM / 方向 | D5、D6、D7 |
| 右电机 PWM / 方向 | D9、D10、D11 |
| OLED SDA / SCL | I2C 总线 |
| 视觉串口输入 | USB Serial |


## 接线说明

- HC-SR04 安装在车头正前方，尽量减少车体结构遮挡。
- 三路红外模块保持同一高度安装，避免阈值漂移过大。
- 编码器脉冲输出接 D2 中断口，用于测速。
- 模式切换按键接 D4，使用内部上拉输入。
- 视觉模块或上位机通过串口发送 `V,error` 格式的偏差值，供视觉模式使用。


## 串口协议

- 视觉模式下输入格式：`V,error`。
- 示例：`V,-12` 表示中心线偏左，小车应向左修正。
- 其它模式下串口主要用于调试观察，不依赖外部持续输入。
- 若超过 `VISION_TIMEOUT_MS` 没有收到新视觉帧，小车会自动停车。


## 运行方式

1. 上传 `src/unmanned_car_perception_and_control_platform/unmanned_car_perception_and_control_platform.ino`。
2. 完成超声波、红外、编码器、电机驱动和 OLED 接线。
3. 先在巡线模式下标定 `LINE_THRESHOLD` 和基础速度。
4. 再切换到避障和速度保持模式验证单模块功能。
5. 最后接入视觉端偏差输入，测试视觉辅助驾驶模式。


## 控制逻辑说明

- `MODE_LINE_FOLLOW` 根据三路红外状态计算误差并输出差速。
- `MODE_OBSTACLE_AVOID` 优先参考超声波距离，小于安全阈值时执行原地转向。
- `MODE_SPEED_HOLD` 利用编码器估算转速，通过调节 PWM 追踪目标转速。
- `MODE_VISION_ASSIST` 依赖视觉偏差串口输入，超时后立即停车保护。
- 所有模式共享同一套电机驱动层，便于后续融合多源决策逻辑。


## 验证标准

| 测试项 | 通过标准 |
| --- | --- |
| 模式切换 | 按键可在四种模式之间循环切换 |
| 巡线控制 | 黑线轨道上可执行基础跟随 |
| 避障控制 | 障碍靠近时小车会转向躲避 |
| 速度保持 | 转速显示会随 PWM 调节趋近目标值 |
| 视觉辅助 | 串口输入偏差后底盘能做左右修正 |


## 可扩展方向

- 增加状态融合与行为决策层。
- 增加车道线识别和摄像头闭环控制。
- 增加 IMU、地图构建和路径规划模块。
- 升级为 ROS 或更完整的自动驾驶实验平台。
